# Qwen2.5 Mini

GQA + RoPE + SwiGLU + RMSNorm, dense and MoE variants.

## Setup

In [1]:
from pathlib import Path
import sys

PACKAGES = Path.cwd()
while not (PACKAGES / "qwen_common").exists():
    if PACKAGES == PACKAGES.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    PACKAGES = PACKAGES.parent
sys.path.insert(0, str(PACKAGES))

import torch

torch.manual_seed(0)

## Dense model

In [2]:
from qwen2_5_mini import Qwen25Model, preset

cfg = preset("demo")
model = Qwen25Model(cfg)
tokens = torch.randint(0, cfg.vocab_size, (2, 8))
logits, info = model(tokens)
print("tokens :", tuple(tokens.shape))
print("logits :", tuple(logits.shape))
print("params :", f"{model.num_parameters():,}")
print("log    :", info["log"][:2])

tokens : (2, 8)
logits : (2, 8, 64)
params : 127,296
log    : ['QwenMini T=8 layers=2 moe=False qk_norm=False', '— layer 0 —']


## MoE variant

The router selects top-k routed experts; shared experts always run.

In [3]:
from qwen2_5_mini import Qwen25MoE

moe_cfg = preset("moe")
moe = Qwen25MoE(moe_cfg)
log = []
y, _ = moe(tokens, log=log)
print("moe logits:", tuple(y.shape))
print("router    :", log[-1])

moe logits: (2, 8, 64)
router    : MoE routed=8 active=2 hist=[3, 1, 0, 1, 8, 7, 5, 7]


## Checkpoint

- What does `n_kv_heads < n_heads` save?
- Why does the MoE have more parameters but a similar forward shape?